## 연습 1: 액세스 제어

당신은 전국 규모의 슈퍼마켓 체인인 **NorthMart Retail**의 데이터 엔지니어입니다. 당신의 팀은 고객 및 거래 데이터를 Unity Catalog에 저장합니다. 분석가가 쿼리를 실행하기 전에, 먼저 카탈로그와 스키마를 만들고 고객 테이블을 채운 다음 — `retail-analysts` 그룹에 필요한 권한을 부여해야 합니다.

> ⚠️ 이 연습을 시작하기 전에 Databricks에서 `retail-analysts` 그룹을 만들고 본인의 사용자 계정을 해당 그룹에 추가했는지 확인하세요. 랩 설정 안내를 참조하세요.

### 작업 1.1 — 카탈로그, 스키마 및 customers 테이블 만들기

아래 셀은 이 연습에 필요한 모든 객체를 설정합니다:
- 모든 NorthMart 데이터 자산을 담을 `retail_catalog` 카탈로그를 만듭니다.
- 그 안에 `security_lab` 스키마를 만듭니다.
- 관리형 Delta 테이블 `retail_catalog.security_lab.customers`를 만들고 6개의 샘플 고객 레코드로 채웁니다. 이 행들은 이 연습의 나머지 부분 전반에서 행 필터링과 열 마스킹을 시연하는 데 사용됩니다.

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

셀을 실행하여 이 객체들을 프로비저닝하세요.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS retail_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'NorthMart Retail 데이터 자산을 위한 카탈로그'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_catalog.security_lab
  COMMENT '보안 랩 연습에 사용되는 스키마';

CREATE OR REPLACE TABLE retail_catalog.security_lab.customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@northmart.com', 'North', 5200.00, 'Standard'),
  (2, 'Bob Martinez',  'bob@northmart.com',   'South', 8100.00, 'Premium'),
  (3, 'Carol Lee',     'carol@northmart.com', 'North', 3400.00, 'Standard'),
  (4, 'David Chen',    'david@northmart.com', 'East',  7800.00, 'Premium'),
  (5, 'Eve Patel',     'eve@northmart.com',   'West',  6200.00, 'Standard'),
  (6, 'Frank Kim',     'frank@northmart.com', 'South', 4100.00, 'Standard')
  AS t(customer_id, customer_name, email, region, total_spend, tier);

### 작업 1.2 — retail-analysts 그룹에 권한 부여

`retail-analysts` 그룹은 `customers` 테이블을 쿼리해야 합니다. Unity Catalog 계층 구조의 필요한 모든 수준에서 필요한 권한을 부여하세요:

- `retail_catalog`에 대한 `USE CATALOG`
- `retail_catalog.security_lab`에 대한 `USE SCHEMA`
- `retail_catalog.security_lab.customers`에 대한 `SELECT`

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks 그룹이 Unity Catalog 테이블에서 SELECT할 수 있도록 허용하려면 어떤 SQL grant가 필요한가요?"*

**힌트:** 다음을 사용하세요: `GRANT <privilege> ON <object type> <object name> TO \`retail-analysts\``.

In [ ]:
%sql
GRANT USE CATALOG ON CATALOG retail_catalog TO `retail-analysts`;
GRANT USE SCHEMA ON SCHEMA retail_catalog.security_lab TO `retail-analysts`;
GRANT SELECT ON TABLE retail_catalog.security_lab.customers TO `retail-analysts`;

### 작업 1.3 — 부여된 권한 확인

권한이 올바르게 할당되었는지 확인하세요. `SHOW GRANTS`를 사용하여 `customers` 테이블의 모든 권한을 표시하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Unity Catalog 테이블의 권한을 확인하기 위해 SHOW GRANTS를 어떻게 사용하나요?"*

**예상 결과:** `retail-analysts`가 `SELECT` 권한과 함께 표시되어야 합니다.

In [ ]:
%sql
SHOW GRANTS ON TABLE retail_catalog.security_lab.customers;

## 연습 2: 행 필터링

NorthMart Retail의 지역 분석가는 자신의 지역에 대한 고객 레코드만 볼 수 있어야 합니다. 테이블을 쿼리하는 사용자에 따라 행을 제한하는 **행 필터 함수**를 구현합니다.

여러 사용자 계정을 만들 수 없으므로, 모든 행을 보는 본인의 사용자 계정을 *제외*한 모든 사람에게 `North` 지역 행만 표시하는 필터를 작성합니다. 이는 지역 분석가가 경험하게 될 것을 시뮬레이션합니다.

### 작업 2.1 — 행 필터 함수 만들기

`STRING` 타입의 `region` 매개변수를 받고 다음 경우 `TRUE`를 반환하는 SQL 함수 `retail_catalog.security_lab.region_filter`를 만드세요:
- `region`이 `'North'`이거나, **또는**
- 현재 사용자가 본인의 계정인 경우(`your@email.com`을 실제 이메일로 바꾸세요).

즉, 권한이 있는 사용자가 아니라면 `North` 지역 행만 표시됩니다.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"액세스를 제어하기 위해 current_user()를 사용하는 행 필터 함수를 Databricks Unity Catalog에서 어떻게 만드나요?"*

**힌트:** 다음을 사용하세요: `CREATE OR REPLACE FUNCTION ... RETURN <boolean expression using current_user()>`.

In [ ]:
%sql
-- 'your@email.com'을 실제 Azure Databricks 사용자 이메일로 바꾸세요
CREATE OR REPLACE FUNCTION retail_catalog.security_lab.region_filter(region STRING)
RETURN region = 'North' OR current_user() = 'your@email.com';

### 작업 2.2 — customers 테이블에 행 필터 적용

`region_filter` 함수를 `customers` 테이블에 연결하여 해당 테이블에 대한 모든 쿼리에 자동으로 적용되도록 하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Azure Databricks에서 Unity Catalog 테이블에 행 필터를 어떻게 설정하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... SET ROW FILTER <function> ON (<column>)`.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers
SET ROW FILTER retail_catalog.security_lab.region_filter ON (region);

### 작업 2.3 — 행 필터 테스트

`customers` 테이블을 쿼리하세요. 필터 함수에서 본인의 이메일이 권한 있는 사용자로 등록되어 있으므로 **6개 행 모두**가 표시되어야 합니다.

다른 사람(본인의 이메일이 아닌)으로 로그인했다면 `North` 지역 행 2개만 보였을 것입니다.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"SQL을 사용하여 Databricks에서 현재 어떤 사용자가 쿼리를 실행 중인지 어떻게 확인할 수 있나요?"*

In [ ]:
%sql
-- 현재 사용자 확인
SELECT current_user();

-- customers 테이블 쿼리 — 권한이 있는 사용자로서 6개 행이 모두 표시되어야 합니다
SELECT * FROM retail_catalog.security_lab.customers;

➡️ 팁: 앞서 만든 행 필터 함수의 이메일 주소를 업데이트한 후 다시 테스트하세요. customers 테이블을 쿼리할 때의 출력을 관찰하세요.

### 작업 2.4 — 행 필터 제거

이제 `customers` 테이블에서 행 필터를 제거하세요. 제거한 후 테이블을 다시 쿼리하여 사용자 컨텍스트와 관계없이 **6개 행 모두**가 반환되는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Unity Catalog 테이블에서 행 필터를 어떻게 제거하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... DROP ROW FILTER`.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers DROP ROW FILTER;

SELECT * FROM retail_catalog.security_lab.customers;

## 연습 3: 열 마스킹

고객 이메일 주소는 PII이므로 권한이 없는 사용자에게는 마스킹되어야 합니다. 대부분의 사용자에게 전체 이메일 주소를 숨기는 **열 마스크 함수**를 만들어 `customers` 테이블의 `email` 열에 적용합니다.

### 작업 3.1 — 열 마스크 함수 만들기

`STRING` 타입의 `email` 매개변수를 받아 다음을 수행하는 SQL 함수 `retail_catalog.security_lab.mask_email`를 만드세요:
- 현재 사용자가 본인의 계정이면 **전체 이메일**을 반환합니다.
- 그 외 모든 사용자에게는 **마스킹된 값**(예: `'***@***.com'`)을 반환합니다.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"실제 값을 반환할지 마스킹된 값을 반환할지 결정하기 위해 current_user()를 사용하는 열 마스킹 함수를 Databricks Unity Catalog에서 어떻게 만드나요?"*

**힌트:** 함수 본문에서 `CASE WHEN current_user() = '...' THEN email ELSE '***@***.com' END` 패턴을 사용하세요.

In [ ]:
%sql
-- 'your@email.com'을 실제 Azure Databricks 사용자 이메일로 바꾸세요
CREATE OR REPLACE FUNCTION retail_catalog.security_lab.mask_email(email STRING)
RETURN CASE
  WHEN current_user() = 'your@email.com' THEN email
  ELSE '***@***.com'
END;

### 작업 3.2 — email 열에 열 마스크 적용

`mask_email` 함수를 `customers` 테이블의 `email` 열에 열 마스크로 적용하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"SQL을 사용하여 Azure Databricks의 Unity Catalog 테이블 열에 열 마스크를 어떻게 설정하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... ALTER COLUMN email SET MASK <function>`.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers
ALTER COLUMN email SET MASK retail_catalog.security_lab.mask_email;

### 작업 3.3 — 열 마스크 테스트

`customers` 테이블을 쿼리하세요. 마스크 함수에 지정된 권한 있는 사용자이므로 **실제 이메일 주소**가 표시되어야 합니다.

이 테이블을 쿼리하는 다른 사용자는 `email` 열에서 `***@***.com`을 보게 됩니다.

In [ ]:
%sql
-- 본인 계정은 실제 이메일을 보고, 다른 사용자는 ***@***.com을 봅니다
SELECT customer_id, customer_name, email, region FROM retail_catalog.security_lab.customers;

### 작업 3.4 — 열 마스크 제거

`email` 열에서 열 마스크를 제거한 다음 테이블을 다시 쿼리하여 이제 모든 사용자에게 이메일이 완전히 보이는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Unity Catalog 테이블 열에서 열 마스크를 어떻게 제거하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... ALTER COLUMN email DROP MASK`.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers ALTER COLUMN email DROP MASK;

SELECT customer_id, customer_name, email, region FROM retail_catalog.security_lab.customers;

## 연습 4: Azure Key Vault 비밀

NorthMart Retail의 로열티 플랫폼은 연결을 위해 API 키가 필요합니다. 이 키를 노트북 내부에 평문으로 저장하는 것은 보안 위험입니다. 이 연습에서는 랩 설정 중에 만든 Azure Key Vault 기반 비밀 스코프에서 API 키를 안전하게 가져옵니다.

> ⚠️ 이 연습의 셀을 실행하기 전에 랩 안내의 Key Vault 설정 단계를 완료했는지 확인하세요(Key Vault 만들기, 비밀 `loyalty-api-key` 추가, 비밀 스코프 `retail-kv-scope` 만들기).

### 작업 4.1 — 스코프에서 사용 가능한 비밀 나열

`retail-kv-scope` 비밀 스코프에서 사용 가능한 모든 비밀을 나열하여 `loyalty-api-key`에 액세스할 수 있는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"dbutils를 사용하여 Databricks 비밀 스코프의 비밀을 어떻게 나열하나요?"*

**힌트:** `dbutils.secrets.list("scope-name")`을 사용하세요. 이는 키 이름만 반환하며 비밀 값은 절대 반환하지 않습니다.

In [ ]:
secrets = dbutils.secrets.list("retail-kv-scope")
for s in secrets:
    print(s.key)

### 작업 4.2 — Python에서 API 키를 안전하게 가져오기

`retail-kv-scope` 스코프에서 `loyalty-api-key` 비밀을 가져와 변수에 할당하세요. 그런 다음 출력하세요.

Azure Databricks가 노트북 출력에서 값을 어떻게 **자동으로 마스킹**하는지 관찰하세요 — 실제 키 대신 `[REDACTED]`가 표시됩니다.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"dbutils.secrets.get을 사용하여 Databricks 비밀 스코프에서 비밀을 어떻게 가져오며 왜 출력이 마스킹되나요?"*

In [ ]:
api_key = dbutils.secrets.get(scope="retail-kv-scope", key="loyalty-api-key")
print(api_key)  # 출력에는 [REDACTED]가 표시됩니다

### 작업 4.3 — 비밀을 사용하여 연결 문자열 시뮬레이션

API 키를 포함하는 연결 문자열을 구성하여 NorthMart의 로열티 플랫폼에 연결하는 방식을 시뮬레이션하세요. 연결 문자열을 출력하고 비밀 값이 여전히 마스킹되어 있는지 관찰하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"값을 노출하지 않고 연결 문자열을 위한 f-string 내부에서 Databricks 비밀을 어떻게 안전하게 사용하나요?"*

In [ ]:
connection_string = f"https://loyalty.northmart.com/api?key={api_key}"
print(connection_string)  # api_key 부분은 [REDACTED]로 표시됩니다

### 작업 4.4 — SQL을 사용하여 비밀 가져오기

`secret()` SQL 함수를 사용하여 동일한 비밀을 가져오세요. 이는 SQL 기반 연결을 구성할 때 유용합니다.

SQL 결과에서도 출력이 마스킹된다는 점에 유의하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"스코프에서 비밀을 가져오기 위해 Databricks SQL의 secret() 함수를 어떻게 사용하나요?"*

In [ ]:
%sql
SELECT secret('retail-kv-scope', 'loyalty-api-key') AS loyalty_api_key;

## 정리

아래 셀을 실행하여 이 랩 중에 만든 모든 객체를 제거하세요.

In [ ]:
%sql
-- 모든 랩 객체 제거
ALTER TABLE retail_catalog.security_lab.customers DROP ROW FILTER;
ALTER TABLE retail_catalog.security_lab.customers ALTER COLUMN email DROP MASK;
DROP TABLE IF EXISTS retail_catalog.security_lab.customers;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.region_filter;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.mask_email;
DROP SCHEMA IF EXISTS retail_catalog.security_lab;
DROP CATALOG IF EXISTS retail_catalog CASCADE;